# Phase 4 - LoRA Fine-Tuning
This notebook clones the repo and runs `src.finetune.train`.

**Pre-flight checklist:**
- Runtime > Change runtime type â†’ **T4 GPU** is selected
- `HF_TOKEN` is set in Colab Secrets (ðŸ”‘ icon in the left sidebar)

In [ ]:
# Cell 1: GPU sanity check â€” must pass before proceeding
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type and select T4 GPU, "
        "then re-run all cells."
    )

print(f"âœ… GPU available: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/SharjilSharma/FineTuningProject.git
%cd FineTuningProject
!pip install -q -r requirements.txt
!pip install -q trl peft bitsandbytes huggingface_hub

In [ ]:
# Cell 3: Load secrets â€” fails loud and fast if HF_TOKEN is missing
from google.colab import userdata
import os

try:
    hf_token = userdata.get('HF_TOKEN')
    if not hf_token:
        raise ValueError("HF_TOKEN secret is empty.")
    os.environ['HF_TOKEN'] = hf_token
    print("âœ… HF_TOKEN loaded.")
except Exception as e:
    raise RuntimeError(
        f"HF_TOKEN not found in Colab Secrets: {e}\n"
        "Add it via the ðŸ”‘ Secrets panel (left sidebar) and re-run."
    ) from e

# Optional: set the HF repo to push the adapter to after training
HF_REPO_ID = "sharjilsharma/earnings-signal-lora-adapter"  # change if needed

In [ ]:
# Cell 4: Run fine-tuning
!python -m src.finetune.train

In [ ]:
# Cell 5: Push LoRA adapter to Hugging Face Hub (private repo)
# This is CRITICAL â€” Colab VMs are ephemeral; anything not pushed is lost on disconnect.
from huggingface_hub import HfApi
from src.finetune.config import FinetuneConfig

cfg = FinetuneConfig()
adapter_dir = cfg.output_dir  # e.g. data/models/finetuned_adapter

api = HfApi(token=os.environ['HF_TOKEN'])

# Create the repo if it doesn't exist yet (private by default)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

api.upload_folder(
    folder_path=adapter_dir,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Phase 4: LoRA adapter trained on bootstrap dataset",
)

print(f"âœ… Adapter pushed to https://huggingface.co/{HF_REPO_ID}")